In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from matplotlib import rcParams
import webbrowser

# ================== تنظیمات فونت و استایل ==================
rcParams['font.family'] = 'DejaVu Sans'
rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

# ================== 0) ورودی‌ها — فقط این 2 مسیر را تغییر دهید ==================
model_ans = pd.read_csv(r"F:\Thesis\project\3-Multi-Agent-System\Test_402\BaseLine\gemini 3 flash\results_few_shot.csv")
gold      = pd.read_csv(r"F:\Thesis\project\3-Multi-Agent-System\Test_402\BaseLine\Gold_402\Gold_402.csv")

model_id_col  = "id"
model_ans_col = "answer"
gold_id_col   = "idx"
gold_ans_col  = "Gold"

# ================== 1) نرمال‌سازی پاسخ‌ها و Gold ==================
def norm_ans(x):
    if pd.isna(x): return None
    s = str(x).strip().strip('"').strip("'")
    try:
        v = int(float(s))
        return str(v) if v in {1, 2, 3, 4} else None
    except:
        for d in ["1", "2", "3", "4"]:
            if s.endswith(d): return d
        return None

def parse_gold_single_or_pair(x):
    if pd.isna(x):
        return set()
    s = str(x).strip()
    if "-" in s:
        parts = [p.strip() for p in s.split("-")]
        return {norm_ans(p) for p in parts if norm_ans(p) is not None}
    v = norm_ans(s)
    return {v} if v is not None else set()

model = model_ans.copy()
gold2 = gold.copy()

model[model_ans_col] = model[model_ans_col].apply(norm_ans)
gold2["gold_set"]     = gold2[gold_ans_col].apply(parse_gold_single_or_pair)
gold2["is_multi"]     = gold2["gold_set"].apply(lambda s: len(s) > 1)
gold2["gold_primary"] = gold2["gold_set"].apply(
    lambda s: sorted(list(s))[0] if len(s) > 0 else None
)

# ================== 2) ادغام ==================
df = pd.merge(
    model,
    gold2[[gold_id_col, "gold_set", "is_multi", "gold_primary"]],
    left_on=model_id_col,
    right_on=gold_id_col,
    how="inner"
)

df["correct"] = df.apply(
    lambda r: (r[model_ans_col] in r["gold_set"]) if r[model_ans_col] is not None else False,
    axis=1
).astype(int)

def answer_status(r):
    if pd.notna(r.get("error", None)) and str(r.get("error", "")).strip():
        return "parse_error"
    if r[model_ans_col] is None:
        return "no_answer"
    return "correct" if r["correct"] else "wrong"

df["status"] = df.apply(answer_status, axis=1)

# ================== 3) پوشه خروجی — کنار همین اسکریپت ==================
script_dir = Path(__file__).parent if "__file__" in dir() else Path.cwd()
root = script_dir / "eval_report"
root.mkdir(parents=True, exist_ok=True)

COLORS = {
    "correct":     "#59A14F",
    "wrong":       "#E15759",
    "no_answer":   "#F28E2B",
    "parse_error": "#BAB0AC",
    "primary":     "#4E79A7",
    "secondary":   "#F28E2B",
}

# ================== 4) eval_accuracy ==================
acc_dir = root / "eval_accuracy"
acc_dir.mkdir(exist_ok=True)

total_q   = len(df)
answered  = df[model_ans_col].notna().sum()
n_correct = df["correct"].sum()
n_wrong   = int((df["correct"] == 0).sum() - df[model_ans_col].isna().sum())
n_na      = int(df[model_ans_col].isna().sum())
acc       = n_correct / total_q * 100 if total_q else 0.0
acc_ans   = n_correct / answered * 100 if answered else 0.0

summary_text = (
    f"Total Questions    : {total_q}\n"
    f"Answered           : {answered}\n"
    f"Correct            : {n_correct}\n"
    f"Wrong              : {n_wrong}\n"
    f"No Answer / Error  : {n_na}\n"
    f"Accuracy (all)     : {acc:.2f}%\n"
    f"Accuracy (answered): {acc_ans:.2f}%\n"
)
(acc_dir / "accuracy_overall.txt").write_text(summary_text, encoding="utf-8")
print(summary_text)

by_pred = (
    df[df[model_ans_col].notna()]
    .groupby(model_ans_col)["correct"]
    .agg(["mean", "count"])
    .reset_index()
)
by_pred["accuracy_%"] = by_pred["mean"] * 100
by_pred.drop(columns=["mean"]).to_csv(acc_dir / "accuracy_by_pred.csv", index=False, encoding="utf-8-sig")

conf = pd.crosstab(
    df[model_ans_col].fillna("N/A"),
    df["gold_primary"],
    rownames=["pred"], colnames=["gold_primary"],
    dropna=False
).fillna(0).astype(int)
conf.to_csv(acc_dir / "confusion.csv", encoding="utf-8-sig")

df_out = df.copy()
df_out["gold_set"] = df_out["gold_set"].apply(lambda s: ",".join(sorted(s)))
df_out.to_csv(acc_dir / "eval_detailed.csv", index=False, encoding="utf-8-sig")

# نمودار 1: Overall Accuracy
fig, ax = plt.subplots(figsize=(4, 4))
bars = ax.bar(["Accuracy (All)", "Accuracy\n(Answered)"], [acc, acc_ans],
               color=[COLORS["primary"], COLORS["secondary"]], width=0.5)
ax.set_ylim(0, 110)
ax.set_ylabel("Accuracy (%)", fontsize=11)
ax.set_title("Overall Accuracy", fontsize=13, fontweight="bold")
for bar, val in zip(bars, [acc, acc_ans]):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 2, f"{val:.1f}%",
            ha="center", va="bottom", fontweight="bold", fontsize=11)
ax.grid(axis="y", alpha=0.4)
plt.tight_layout()
plt.savefig(acc_dir / "plot_accuracy_overall.png", dpi=200, bbox_inches="tight", facecolor="white")
plt.close()

# نمودار 2: Accuracy by Predicted Option
tmp = by_pred.sort_values(model_ans_col)
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(tmp[model_ans_col].astype(str), tmp["accuracy_%"],
               color=COLORS["secondary"], edgecolor="white", linewidth=1.5)
ax.set_ylim(0, 110)
ax.set_xlabel("Predicted Option", fontsize=11)
ax.set_ylabel("Accuracy (%)", fontsize=11)
ax.set_title("Accuracy by Predicted Option", fontsize=13, fontweight="bold")
for bar, (_, row) in zip(bars, tmp.iterrows()):
    ax.text(bar.get_x() + bar.get_width() / 2, row["accuracy_%"] + 2,
            f"{row['accuracy_%']:.0f}%\n(n={int(row['count'])})",
            ha="center", va="bottom", fontsize=9)
ax.grid(axis="y", alpha=0.4)
plt.tight_layout()
plt.savefig(acc_dir / "plot_accuracy_by_pred.png", dpi=200, bbox_inches="tight", facecolor="white")
plt.close()

# نمودار 3: Status Distribution (Donut)
status_counts = df["status"].value_counts()
colors_pie = [COLORS.get(s, "#999") for s in status_counts.index]
fig, ax = plt.subplots(figsize=(5, 5))
wedges, texts, autotexts = ax.pie(
    status_counts.values, labels=status_counts.index,
    autopct="%1.1f%%", colors=colors_pie,
    startangle=90, pctdistance=0.75,
    wedgeprops=dict(width=0.55, edgecolor="white", linewidth=2)
)
for t in autotexts: t.set_fontsize(10)
ax.set_title("Answer Status Distribution", fontsize=13, fontweight="bold", pad=20)
plt.tight_layout()
plt.savefig(acc_dir / "plot_status_donut.png", dpi=200, bbox_inches="tight", facecolor="white")
plt.close()

# نمودار 4: Confusion Heatmap
fig, ax = plt.subplots(figsize=(max(4, len(conf.columns)+1), max(3, len(conf)+1)))
sns.heatmap(conf, annot=True, fmt="d", cmap="Blues", cbar=False,
            linewidths=1, linecolor="white", ax=ax)
ax.set_title("Confusion Matrix  (pred × gold_primary)", fontsize=12, fontweight="bold")
ax.set_xlabel("Gold (primary)", fontsize=11)
ax.set_ylabel("Predicted", fontsize=11)
plt.tight_layout()
plt.savefig(acc_dir / "plot_confusion_heatmap.png", dpi=200, bbox_inches="tight", facecolor="white")
plt.close()
print("✅ eval_accuracy done")

# ================== 5) tokens_usage ==================
tok_dir = root / "tokens_usage"
tok_dir.mkdir(exist_ok=True)
tok_cols = [c for c in ["prompt_tokens", "completion_tokens", "total_tokens"] if c in df.columns]

if tok_cols:
    df[tok_cols].agg(["mean", "median", "min", "max"]).round(2).to_csv(
        tok_dir / "tokens_summary.csv", encoding="utf-8-sig")
    df.groupby("correct")[tok_cols].mean().round(2).to_csv(
        tok_dir / "tokens_by_correct.csv", encoding="utf-8-sig")

    if "completion_tokens" in df.columns and df["completion_tokens"].notna().any():
        fig, ax = plt.subplots(figsize=(6, 4))
        ax.hist(df["completion_tokens"].dropna(), bins=15,
                color=COLORS["correct"], edgecolor="white", linewidth=1.2)
        ax.axvline(df["completion_tokens"].mean(), color="#E15759", linestyle="--",
                   linewidth=1.5, label=f"Mean: {df['completion_tokens'].mean():.0f}")
        ax.set_xlabel("Completion Tokens", fontsize=11)
        ax.set_ylabel("Frequency", fontsize=11)
        ax.set_title("Completion Tokens Distribution", fontsize=13, fontweight="bold")
        ax.legend(); ax.grid(axis="y", alpha=0.4)
        plt.tight_layout()
        plt.savefig(tok_dir / "plot_tokens_hist.png", dpi=200, bbox_inches="tight", facecolor="white")
        plt.close()

    if ("completion_tokens" in df.columns and "latency_ms" in df.columns
            and df["completion_tokens"].notna().any() and df["latency_ms"].notna().any()):
        fig, ax = plt.subplots(figsize=(6, 4))
        c_map = df["correct"].map({1: COLORS["correct"], 0: COLORS["wrong"]})
        ax.scatter(df["completion_tokens"], df["latency_ms"],
                   c=c_map, alpha=0.75, s=80, edgecolors="white", linewidths=0.8)
        ax.set_xlabel("Completion Tokens", fontsize=11)
        ax.set_ylabel("Latency (ms)", fontsize=11)
        ax.set_title("Completion Tokens vs Latency", fontsize=13, fontweight="bold")
        patch_c = mpatches.Patch(color=COLORS["correct"], label="Correct")
        patch_w = mpatches.Patch(color=COLORS["wrong"],   label="Wrong")
        ax.legend(handles=[patch_c, patch_w])
        ax.grid(alpha=0.3)
        plt.tight_layout()
        plt.savefig(tok_dir / "plot_tokens_vs_latency.png", dpi=200, bbox_inches="tight", facecolor="white")
        plt.close()
print("✅ tokens_usage done")

# ================== 6) explanation_quality ==================
exp_dir = root / "explanation_quality"
exp_dir.mkdir(exist_ok=True)

if "explain_len_words" in df.columns and df["explain_len_words"].notna().any():
    summary_exp = (
        df.groupby("correct")["explain_len_words"]
        .agg(["mean", "median", "min", "max", "count"])
        .round(2).reset_index()
    )
    summary_exp["correct"] = summary_exp["correct"].map({0: "Wrong", 1: "Correct"})
    summary_exp.to_csv(exp_dir / "explain_len_vs_correct.csv", index=False, encoding="utf-8-sig")

    fig, ax = plt.subplots(figsize=(5, 4))
    df_box = df[df["explain_len_words"].notna()].copy()
    df_box["Correctness"] = df_box["correct"].map({0: "Wrong", 1: "Correct"})
    sns.boxplot(data=df_box, x="Correctness", y="explain_len_words", hue="Correctness",
                palette={"Wrong": COLORS["wrong"], "Correct": COLORS["correct"]},
                width=0.45, linewidth=1.5, legend=False, ax=ax)
    ax.set_xlabel(""); ax.set_ylabel("Explanation Length (words)", fontsize=11)
    ax.set_title("Explanation Length vs Correctness", fontsize=13, fontweight="bold")
    ax.grid(axis="y", alpha=0.4)
    plt.tight_layout()
    plt.savefig(exp_dir / "plot_explain_len_box.png", dpi=200, bbox_inches="tight", facecolor="white")
    plt.close()
print("✅ explanation_quality done")

# ================== 7) latency_profile ==================
lat_dir = root / "latency_profile"
lat_dir.mkdir(exist_ok=True)

if "latency_ms" in df.columns and df["latency_ms"].notna().any():
    lat_s = df["latency_ms"].dropna()
    lat_summary = pd.Series({
        "mean":   lat_s.mean(),
        "median": lat_s.median(),
        "min":    lat_s.min(),
        "max":    lat_s.max(),
        "p90":    np.percentile(lat_s, 90),
        "p95":    np.percentile(lat_s, 95),
    }).round(1)
    lat_summary.to_csv(lat_dir / "latency_summary.csv", header=["value"], encoding="utf-8-sig")

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.hist(lat_s, bins=15, color=COLORS["primary"], edgecolor="white", linewidth=1.2)
    ax.axvline(lat_s.mean(), color=COLORS["wrong"], linestyle="--", linewidth=1.5,
               label=f"Mean: {lat_s.mean():.0f} ms")
    ax.axvline(np.percentile(lat_s, 90), color="#F28E2B", linestyle=":", linewidth=1.5,
               label=f"P90: {np.percentile(lat_s,90):.0f} ms")
    ax.set_xlabel("Latency (ms)", fontsize=11); ax.set_ylabel("Frequency", fontsize=11)
    ax.set_title("Latency Distribution", fontsize=13, fontweight="bold")
    ax.legend(); ax.grid(axis="y", alpha=0.4)
    plt.tight_layout()
    plt.savefig(lat_dir / "plot_latency_dist.png", dpi=200, bbox_inches="tight", facecolor="white")
    plt.close()

    fig, ax = plt.subplots(figsize=(5, 4))
    df_lat = df[df["latency_ms"].notna()].copy()
    df_lat["Correctness"] = df_lat["correct"].map({0: "Wrong", 1: "Correct"})
    sns.boxplot(data=df_lat, x="Correctness", y="latency_ms", hue="Correctness",
                palette={"Wrong": COLORS["wrong"], "Correct": COLORS["correct"]},
                width=0.45, linewidth=1.5, legend=False, ax=ax)
    ax.set_ylabel("Latency (ms)", fontsize=11); ax.set_xlabel("")
    ax.set_title("Latency by Correctness", fontsize=13, fontweight="bold")
    ax.grid(axis="y", alpha=0.4)
    plt.tight_layout()
    plt.savefig(lat_dir / "plot_latency_by_correct.png", dpi=200, bbox_inches="tight", facecolor="white")
    plt.close()
print("✅ latency_profile done")

# ================== 8) calibration_confidence ==================
cal_dir = root / "calibration_confidence"
cal_dir.mkdir(exist_ok=True)

if "confidence" in df.columns and df["confidence"].notna().any():
    calib = df.groupby("confidence")["correct"].agg(["mean", "count"]).reset_index()
    calib.rename(columns={"mean": "accuracy"}, inplace=True)
    calib["accuracy_%"] = calib["accuracy"] * 100
    calib.to_csv(cal_dir / "calibration_by_conf.csv", index=False, encoding="utf-8-sig")

    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(calib["confidence"], calib["accuracy_%"], marker="o", color=COLORS["primary"], label="Observed")
    ax.set_xticks([1,2,3,4,5]); ax.set_ylim(0, 100)
    ax.set_xlabel("Confidence"); ax.set_ylabel("Accuracy (%)"); ax.set_title("Reliability Curve")
    ax.legend()
    plt.tight_layout()
    plt.savefig(cal_dir / "plot_reliability_curve.png", dpi=200, bbox_inches="tight", facecolor="white")
    plt.close()

    rows = []
    for t in [1, 2, 3, 4, 5]:
        sub = df[df["confidence"] >= t]
        coverage = len(sub) / len(df) * 100 if len(df) else 0.0
        acc_t = sub["correct"].mean() * 100 if len(sub) else np.nan
        rows.append({"threshold": t, "coverage_%": coverage, "accuracy_%": acc_t, "n": len(sub)})
    pd.DataFrame(rows).to_csv(cal_dir / "threshold_tradeoff.csv", index=False, encoding="utf-8-sig")
print("✅ calibration_confidence done")

# ================== 9) error_inspection ==================
err_dir = root / "error_inspection"
err_dir.mkdir(exist_ok=True)

err_pairs = (
    conf.stack().reset_index(name="count")
    .rename(columns={"pred": "predicted", "gold_primary": "gold"})
)
err_pairs = err_pairs[err_pairs["predicted"] != err_pairs["gold"].astype(str)]
err_pairs = err_pairs[err_pairs["count"] > 0].sort_values("count", ascending=False)
err_pairs.to_csv(err_dir / "top_mistakes.csv", index=False, encoding="utf-8-sig")

df_wrg = df[df["correct"] == 0].copy()
df_wrg["gold_set"] = df_wrg["gold_set"].apply(lambda s: ",".join(sorted(s)))
df_wrg.to_csv(err_dir / "all_wrong_answers.csv", index=False, encoding="utf-8-sig")

df_na2 = df[df[model_ans_col].isna()].copy()
df_na2["gold_set"] = df_na2["gold_set"].apply(lambda s: ",".join(sorted(s)))
df_na2.to_csv(err_dir / "no_answer_questions.csv", index=False, encoding="utf-8-sig")

if "confidence" in df.columns and df["confidence"].notna().any():
    hf = df[(df["correct"] == 0) & (df["confidence"] >= 4)].copy()
    lt = df[(df["correct"] == 1) & (df["confidence"] <= 2)].copy()
    hf["gold_set"] = hf["gold_set"].apply(lambda s: ",".join(sorted(s)))
    lt["gold_set"] = lt["gold_set"].apply(lambda s: ",".join(sorted(s)))
    hf.to_csv(err_dir / "review_hard_false_high_conf.csv", index=False, encoding="utf-8-sig")
    lt.to_csv(err_dir / "review_lucky_true_low_conf.csv",  index=False, encoding="utf-8-sig")
print("✅ error_inspection done")

# ================== 10) گزارش HTML نهایی ==================
model_name = df["model"].iloc[0] if "model" in df.columns else "Unknown Model"

rows_html = ""
for _, r in df.iterrows():
    status_color = {
        "correct":     "#d4edda",
        "wrong":       "#f8d7da",
        "no_answer":   "#fff3cd",
        "parse_error": "#f5f5f5"
    }.get(r["status"], "white")
    gold_disp = r["gold_set"] if isinstance(r["gold_set"], str) else ",".join(sorted(r["gold_set"]))
    pred_disp = r[model_ans_col] if r[model_ans_col] else "—"
    explanation_short = (str(r.get("explanation", ""))[:150] + "...") if pd.notna(r.get("explanation")) else "—"
    rows_html += f"""
    <tr style="background:{status_color}">
      <td>{r['id']}</td>
      <td><b>{pred_disp}</b></td>
      <td><b>{gold_disp}</b></td>
      <td style="text-align:center">{'✅' if r['correct'] else '❌'}</td>
      <td>{r.get('latency_ms','—')}</td>
      <td>{int(r['completion_tokens']) if pd.notna(r.get('completion_tokens')) else '—'}</td>
      <td style="font-size:12px;max-width:320px;direction:rtl">{explanation_short}</td>
    </tr>"""

html_content = f"""<!DOCTYPE html>
<html lang="fa" dir="rtl">
<head>
<meta charset="UTF-8">
<title>Evaluation Report – {model_name}</title>
<style>
  body {{ font-family: Tahoma, Arial, sans-serif; background:#f8f9fa; color:#333; margin:0; padding:24px; direction:rtl }}
  h1 {{ color:#2c3e50; border-bottom:3px solid #4E79A7; padding-bottom:10px; margin-bottom:6px }}
  h2 {{ color:#4E79A7; margin-top:32px; margin-bottom:12px }}
  .kpi-grid {{ display:grid; grid-template-columns:repeat(auto-fit,minmax(150px,1fr)); gap:14px; margin:16px 0 }}
  .kpi-card {{ background:white; border-radius:10px; padding:16px; text-align:center; box-shadow:0 2px 8px rgba(0,0,0,.07) }}
  .kpi-card .val {{ font-size:1.9em; font-weight:bold; color:#4E79A7 }}
  .kpi-card .lbl {{ font-size:.82em; color:#666; margin-top:4px }}
  table {{ width:100%; border-collapse:collapse; background:white; border-radius:8px; overflow:hidden; box-shadow:0 2px 8px rgba(0,0,0,.07); font-size:13px }}
  th {{ background:#4E79A7; color:white; padding:10px 8px; font-weight:600; text-align:right }}
  td {{ padding:8px; border-bottom:1px solid #eee; text-align:right; vertical-align:top }}
  .img-grid {{ display:grid; grid-template-columns:repeat(auto-fit,minmax(320px,1fr)); gap:18px; margin:16px 0 }}
  .img-card {{ background:white; border-radius:10px; padding:14px; box-shadow:0 2px 8px rgba(0,0,0,.07) }}
  .img-card img {{ width:100%; border-radius:6px }}
  .img-card p {{ margin:8px 0 0; font-size:.83em; color:#777; text-align:center }}
  footer {{ margin-top:40px; text-align:center; color:#aaa; font-size:.82em }}
</style>
</head>
<body>
<h1>📊 گزارش ارزیابی مدل — {model_name}</h1>
<p style="color:#888;margin-top:2px">سوالات: {total_q} | پاسخ‌داده‌شده: {int(answered)} | بی‌پاسخ/خطا: {n_na}</p>

<h2>🎯 خلاصه عملکرد</h2>
<div class="kpi-grid">
  <div class="kpi-card"><div class="val">{total_q}</div><div class="lbl">کل سوالات</div></div>
  <div class="kpi-card"><div class="val" style="color:#59A14F">{n_correct}</div><div class="lbl">پاسخ صحیح</div></div>
  <div class="kpi-card"><div class="val" style="color:#E15759">{n_wrong}</div><div class="lbl">پاسخ غلط</div></div>
  <div class="kpi-card"><div class="val" style="color:#F28E2B">{n_na}</div><div class="lbl">بی‌پاسخ / خطا</div></div>
  <div class="kpi-card"><div class="val">{acc:.1f}%</div><div class="lbl">دقت کل</div></div>
  <div class="kpi-card"><div class="val">{acc_ans:.1f}%</div><div class="lbl">دقت پاسخ‌داده‌شده</div></div>
  <div class="kpi-card"><div class="val">{df['latency_ms'].mean():.0f} ms</div><div class="lbl">میانگین تأخیر</div></div>
  <div class="kpi-card"><div class="val">{df['completion_tokens'].mean():.0f}</div><div class="lbl">میانگین توکن خروجی</div></div>
</div>

<h2>📈 نمودارها</h2>
<div class="img-grid">
  <div class="img-card"><img src="eval_accuracy/plot_accuracy_overall.png"><p>دقت کلی مدل</p></div>
  <div class="img-card"><img src="eval_accuracy/plot_accuracy_by_pred.png"><p>دقت به تفکیک گزینه</p></div>
  <div class="img-card"><img src="eval_accuracy/plot_status_donut.png"><p>توزیع وضعیت پاسخ‌ها</p></div>
  <div class="img-card"><img src="eval_accuracy/plot_confusion_heatmap.png"><p>ماتریس درهم‌ریختگی</p></div>
  <div class="img-card"><img src="tokens_usage/plot_tokens_hist.png"><p>توزیع توکن‌های خروجی</p></div>
  <div class="img-card"><img src="tokens_usage/plot_tokens_vs_latency.png"><p>توکن vs تأخیر</p></div>
  <div class="img-card"><img src="explanation_quality/plot_explain_len_box.png"><p>طول توضیح vs صحت</p></div>
  <div class="img-card"><img src="latency_profile/plot_latency_dist.png"><p>توزیع تأخیر</p></div>
  <div class="img-card"><img src="latency_profile/plot_latency_by_correct.png"><p>تأخیر به تفکیک صحت</p></div>
</div>

<h2>📋 جزئیات پاسخ‌ها</h2>
<table>
<thead><tr>
  <th>ID</th><th>پاسخ مدل</th><th>پاسخ صحیح</th><th>نتیجه</th>
  <th>تأخیر (ms)</th><th>توکن</th><th>خلاصه استدلال</th>
</tr></thead>
<tbody>{rows_html}</tbody>
</table>
<footer>Eval Pipeline · {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')}</footer>
</body>
</html>"""

report_path = root / "report.html"
report_path.write_text(html_content, encoding="utf-8")

print(f"\n✅ همه گزارش‌ها در پوشه زیر ذخیره شدند:")
print(f"   {root.resolve()}")
print(f"\n📄 گزارش HTML: {report_path.resolve()}")

# باز کردن خودکار گزارش در مرورگر
webbrowser.open(report_path.resolve().as_uri())
print("\n🌐 گزارش در مرورگر باز شد.")

Total Questions    : 10
Answered           : 10
Correct            : 8
Wrong              : 2
No Answer / Error  : 0
Accuracy (all)     : 80.00%
Accuracy (answered): 80.00%

✅ eval_accuracy done
✅ tokens_usage done
✅ explanation_quality done
✅ latency_profile done
✅ calibration_confidence done
✅ error_inspection done

✅ همه گزارش‌ها در پوشه زیر ذخیره شدند:
   F:\Thesis\project\3-Multi-Agent-System\Test_402\BaseLine\gemini 3 flash\eval\eval_report

📄 گزارش HTML: F:\Thesis\project\3-Multi-Agent-System\Test_402\BaseLine\gemini 3 flash\eval\eval_report\report.html

🌐 گزارش در مرورگر باز شد.
